# Phase 3B EEG Preprocessing: Focused Quality Review

This notebook executes the **Phase 3B Preprocessing Quality Review** (`imagined_speech.data.preprocessing_review`) on the completed Phase 3B preprocessing outputs across all 690 EDF recordings.

### Scientific Safeguards & Review Objectives
1. **Identification of Extreme Traces**: Identifies recordings and channels with the highest post-filter peak-to-peak amplitudes, the highest boundary-to-interior standard-deviation ratios, the five lowest standard deviations, and the largest residual channel means.
2. **Detailed Multi-Panel Visualizations**: Produces plots of representative extreme traces showing the full waveform, initial/final 1-second boundary segments, and interior, with axes and units labeled in µV and seconds.
3. **Diagnostic Nature of Boundary Ratios**: Boundary-to-interior standard-deviation ratios are evaluated strictly as **exploratory diagnostic indicators** of edge variance (e.g., ocular blinks or movement transients near recording boundaries), NOT confirmed artifacts or proof of filter failure, to guide downstream windowing and trimming margins.
4. **Concentration Analysis**: Evaluates whether extreme values concentrate in particular tasks, channels, or filename prefixes. Filename prefixes (`A`–`W`) are treated strictly as **unverified candidate metadata**; no participant-identity assumptions are made.
5. **Exploratory Screening Confirmation**: Confirms configured screening thresholds (`min_std_uV = 0.5`, `max_ptp_uV = 1000.0`) and explains that flags are exploratory screening indicators, not verified physiological or hardware artifacts.
6. **100% Data Retention**: Adds no automatic rejection, channel interpolation, spatial rereferencing (CAR remains non-default), normalization, windowing/epoching, train/test splitting, feature extraction, or model training.
7. **Three-Recording Smoke Test**: Executes a 3-recording smoke review (one each from `Char`, `Digit`, and `Image`) in a temporary directory to validate end-to-end functionality before processing the full dataset.
8. **Strict Integrity Verification & Limitation Reporting**: Computes SHA-256 hashes of file contents before and after the review across raw EDFs and all existing Phase 1, Phase 2, Phase 3A, and Phase 3B artifacts. If any required baseline snapshot is unavailable, reports that limitation explicitly rather than claiming full integrity verification.
9. **Review Outputs**: Saves review tables, JSON summary, Markdown report, and figures to `/content/drive/MyDrive/imagined_speech_outputs/phase3b_preprocessing_review/` in Colab (or `./outputs/phase3b_preprocessing_review/` locally).

## 1. Mount Google Drive

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Google Colab: {IN_COLAB}')

## 2. Clone or Update the Repository

In [ ]:
import subprocess

REPO_URL = 'https://github.com/ACT2039/imagined-speech-eeg.git'
COLAB_REPO_DIR = Path('/content/imagined-speech-eeg')

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        print(f'Cloning {REPO_URL} into {COLAB_REPO_DIR}...')
        subprocess.run(['git', 'clone', REPO_URL, str(COLAB_REPO_DIR)], check=True)
    elif (COLAB_REPO_DIR / '.git').is_dir():
        print(f'Updating existing repository at {COLAB_REPO_DIR}...')
        subprocess.run(['git', '-C', str(COLAB_REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        raise RuntimeError(
            f'Directory {COLAB_REPO_DIR} exists but is not a valid Git repository.'
        )
    os.chdir(COLAB_REPO_DIR)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError(
        f'pyproject.toml not found in working directory {REPO_ROOT}. '
        'Ensure the notebook is running from the repository root.'
    )

print(f'Repository root: {REPO_ROOT}')

## 3. Install the Package

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)

src_dir = str((REPO_ROOT / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import imagined_speech
print(f'Installed imagined_speech version: {imagined_speech.__version__}')

## 4. Resolve `DATA_ROOT` and Directory Paths

In [ ]:
import json
import pandas as pd
from IPython.display import display

CONFIRMED_COLAB_DATA_ROOT = Path('/content/data/raw/Imagined_speech_EEG_edf')
EXTRACT_ROOT = Path('/content/data/raw' if IN_COLAB else './data/raw')

if os.environ.get('ISR_DATA_ROOT'):
    DATA_ROOT = Path(os.environ['ISR_DATA_ROOT']).resolve()
elif IN_COLAB and CONFIRMED_COLAB_DATA_ROOT.is_dir():
    DATA_ROOT = CONFIRMED_COLAB_DATA_ROOT.resolve()
else:
    DATA_ROOT = EXTRACT_ROOT.resolve()

PHASE1_AUDIT_DIR = Path(
    os.environ.get(
        'ISR_PHASE1_AUDIT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase1_audit'
        if IN_COLAB
        else './outputs/phase1_audit',
    )
)
PHASE2_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_PHASE2_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase2_signal_quality'
        if IN_COLAB
        else './outputs/phase2_signal_quality',
    )
)
PHASE3A_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_PHASE3A_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase3a_diagnostic'
        if IN_COLAB
        else './outputs/phase3a_diagnostic',
    )
)
PHASE3B_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_PHASE3B_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase3b_preprocessing'
        if IN_COLAB
        else './outputs/phase3b_preprocessing',
    )
)
REVIEW_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase3b_preprocessing_review'
        if IN_COLAB
        else './outputs/phase3b_preprocessing_review',
    )
)

required_p3b_files = ('preprocessed_manifest.csv', 'channel_qc_metrics.csv', 'phase3b_preprocessing_summary.json')
missing_p3b = [f for f in required_p3b_files if not (PHASE3B_OUTPUT_DIR / f).is_file()]
if missing_p3b:
    raise FileNotFoundError(
        f'Required Phase 3B outputs are missing in {PHASE3B_OUTPUT_DIR}: {missing_p3b}. '
        'Please run notebooks/04_phase3b_preprocessing.ipynb first.'
    )

print(f'Resolved DATA_ROOT:        {DATA_ROOT}')
print(f'Resolved Phase 1 dir:     {PHASE1_AUDIT_DIR}')
print(f'Resolved Phase 2 dir:     {PHASE2_OUTPUT_DIR}')
print(f'Resolved Phase 3A dir:    {PHASE3A_OUTPUT_DIR}')
print(f'Resolved Phase 3B dir:    {PHASE3B_OUTPUT_DIR}')
print(f'Target Review dir:        {REVIEW_OUTPUT_DIR}')

## 5. Compute Pre-Review SHA-256 Baseline Snapshots
Hashes all file contents across raw EDFs and Phase 1, Phase 2, Phase 3A, and Phase 3B artifacts. If any directory is unavailable or empty, reports the limitation explicitly.

In [ ]:
from imagined_speech.data.preprocessing_review import compute_directory_sha256_snapshot

print('Computing pre-review SHA-256 snapshots...')
raw_edf_hashes_before = compute_directory_sha256_snapshot(DATA_ROOT, pattern='*.edf')
phase1_hashes_before = compute_directory_sha256_snapshot(PHASE1_AUDIT_DIR)
phase2_hashes_before = compute_directory_sha256_snapshot(PHASE2_OUTPUT_DIR)
phase3a_hashes_before = compute_directory_sha256_snapshot(PHASE3A_OUTPUT_DIR)
phase3b_hashes_before = compute_directory_sha256_snapshot(PHASE3B_OUTPUT_DIR)

baseline_snapshots = {
    'raw_edfs': raw_edf_hashes_before,
    'phase1_audit': phase1_hashes_before,
    'phase2_signal_quality': phase2_hashes_before,
    'phase3a_diagnostic': phase3a_hashes_before,
    'phase3b_preprocessing': phase3b_hashes_before,
}

for stage_name, hashes in baseline_snapshots.items():
    if hashes:
        print(f'[BASELINE AVAILABLE] {stage_name}: {len(hashes)} files snapshotted via SHA-256.')
    else:
        print(f'[LIMITATION] Baseline snapshot for {stage_name} is unavailable. Full byte-level integrity verification cannot be performed for this stage.')

## 6. Run Quality Review Unit Tests

In [ ]:
print('Running unit tests for Phase 3B quality review...')
pytest_proc = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_preprocessing_review.py', '-v'],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
    check=False,
)
if pytest_proc.stdout:
    print(pytest_proc.stdout)
if pytest_proc.stderr:
    print(pytest_proc.stderr, file=sys.stderr)
if pytest_proc.returncode != 0:
    raise RuntimeError(
        f'Quality review unit tests failed with exit code {pytest_proc.returncode}. '
        'See captured pytest output above.'
    )
print('Quality review unit tests passed successfully!')

## 7. Phase 3B Quality Review Smoke Test (3 Recordings: 1 each from Char, Digit, Image)
Executes a smoke review into a temporary directory using one recording each from `Char`, `Digit`, and `Image` before running the full dataset review.

In [ ]:
import shutil
import tempfile
from imagined_speech.data.preprocessing_review import run_phase3b_preprocessing_review

manifest_full = pd.read_csv(PHASE3B_OUTPUT_DIR / 'preprocessed_manifest.csv')
qc_full = pd.read_csv(PHASE3B_OUTPUT_DIR / 'channel_qc_metrics.csv')

smoke_recs = []
for task_name in ('Char', 'Digit', 'Image'):
    sub = manifest_full[manifest_full['task_category'] == task_name]
    if not sub.empty:
        smoke_recs.append(str(sub.iloc[0]['relative_source_path']))
print(f'Selected 3 representative smoke recordings (1 per task): {smoke_recs}')

with tempfile.TemporaryDirectory(prefix='phase3b_review_smoke_') as smoke_tmp:
    smoke_tmp_p = Path(smoke_tmp)
    smoke_p3b_dir = smoke_tmp_p / 'preprocessed'
    smoke_out_dir = smoke_tmp_p / 'review_out'
    smoke_p3b_dir.mkdir(parents=True)

    # Subset manifest and channel QC records for the 3 recordings
    smoke_manifest = manifest_full[manifest_full['relative_source_path'].isin(smoke_recs)].copy()
    smoke_qc = qc_full[qc_full['relative_source_path'].isin(smoke_recs)].copy()
    smoke_manifest.to_csv(smoke_p3b_dir / 'preprocessed_manifest.csv', index=False)
    smoke_qc.to_csv(smoke_p3b_dir / 'channel_qc_metrics.csv', index=False)

    # Copy summary JSON and the 3 corresponding .npz signal files
    summary_src = PHASE3B_OUTPUT_DIR / 'phase3b_preprocessing_summary.json'
    if summary_src.is_file():
        shutil.copy(summary_src, smoke_p3b_dir / 'phase3b_preprocessing_summary.json')
    for rel_path in smoke_recs:
        src_npz = PHASE3B_OUTPUT_DIR / 'signals' / Path(rel_path).with_suffix('.npz')
        if src_npz.is_file():
            dst_npz = smoke_p3b_dir / 'signals' / Path(rel_path).with_suffix('.npz')
            dst_npz.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(src_npz, dst_npz)

    # Execute smoke review
    smoke_res = run_phase3b_preprocessing_review(
        preprocessed_dir=smoke_p3b_dir,
        output_dir=smoke_out_dir,
        save_outputs=True,
        generate_figures=True,
    )
    assert smoke_res.summary['n_recordings_inspected'] == 3
    assert smoke_res.summary['n_channel_traces_audited'] == 42
    assert (smoke_out_dir / 'phase3b_review_report.md').is_file()
    assert (smoke_out_dir / 'phase3b_review_summary.json').is_file()
    print('Phase 3B 3-recording smoke review completed and validated successfully!')

## 8. Execute Full Dataset Phase 3B Quality Review (All 690 Recordings)
Runs `run_phase3b_preprocessing_review` across all 690 preprocessed recordings, writing summary reports, tables, and figures to `REVIEW_OUTPUT_DIR`.

In [ ]:
review_result = run_phase3b_preprocessing_review(
    preprocessed_dir=PHASE3B_OUTPUT_DIR,
    output_dir=REVIEW_OUTPUT_DIR,
    raw_edf_dir=DATA_ROOT,
    phase1_dir=PHASE1_AUDIT_DIR,
    phase2_dir=PHASE2_OUTPUT_DIR,
    phase3a_dir=PHASE3A_OUTPUT_DIR,
    baseline_snapshots=baseline_snapshots,
    save_outputs=True,
    generate_figures=True,
)
print('=== Phase 3B Quality Review Summary ===')
print(json.dumps(review_result.summary, indent=2))

## 9. Review Extreme Traces and Residual Channel Means
> **Important Diagnostic Clarification**: Elevated boundary-to-interior standard deviation ratios (>2.0x) are strictly exploratory diagnostic indicators of edge variance (e.g. blinks or movement near trial onset/offset), NOT confirmed artifacts or proof of filter failure. They guide downstream epoch margin selection without rejecting or modifying any signals.

In [ ]:
print('=== Top 10 Extreme Post-Filter Peak-to-Peak (PTP) Channel Traces ===')
display(
    review_result.top_ptp_df[
        [
            'task_category',
            'normalized_label',
            'unverified_candidate_prefix',
            'channel_name',
            'post_filter_ptp_uV',
            'post_filter_std_uV',
            'post_filter_boundary_to_interior_std_ratio',
            'relative_source_path',
        ]
    ].head(10)
)

print('\n=== Top 10 Boundary-to-Interior Std Ratio Channel Traces ===')
display(
    review_result.top_boundary_ratio_df[
        [
            'task_category',
            'normalized_label',
            'unverified_candidate_prefix',
            'channel_name',
            'post_filter_boundary_to_interior_std_ratio',
            'post_filter_boundary_1s_std_uV',
            'post_filter_interior_std_uV',
            'post_filter_ptp_uV',
            'relative_source_path',
        ]
    ].head(10)
)

print('\n=== Five Lowest Post-Filter Standard Deviation Channel Traces ===')
display(
    review_result.lowest_std_df[
        [
            'task_category',
            'normalized_label',
            'unverified_candidate_prefix',
            'channel_name',
            'post_filter_std_uV',
            'post_filter_ptp_uV',
            'pre_filter_std_uV',
            'screening_flag_low_std',
            'relative_source_path',
        ]
    ].head(5)
)

print('\n=== Largest Absolute Residual Channel Means (DC Removal Verification) ===')
display(
    review_result.largest_residual_mean_df[
        [
            'task_category',
            'normalized_label',
            'unverified_candidate_prefix',
            'channel_name',
            'post_filter_mean_uV',
            'abs_post_filter_mean_uV',
            'subtracted_dc_mean_uV',
            'relative_source_path',
        ]
    ].head(10)
)

## 10. Review Concentration Across Tasks, Channels, and Candidate Prefixes
> **Unverified Metadata Note**: Filename prefixes (`A`–`W`) are treated strictly as unverified metadata. Because EDF patient headers are anonymized/blank (`X X X X`), prefixes cannot be assumed to represent distinct human participants without external protocol confirmation.

In [ ]:
print('=== Concentration by Task Category ===')
display(review_result.concentration_by_task_df)

print('\n=== Concentration by EEG Channel Montage ===')
display(review_result.concentration_by_channel_df)

print('\n=== Concentration by Candidate Prefix (Top 10, Unverified Metadata) ===')
display(review_result.concentration_by_prefix_df.head(10))

## 11. Post-Review Checksum Audit, Markdown Report, and Figures

In [ ]:
from IPython.display import Image, Markdown
from imagined_speech.data.preprocessing_review import verify_directory_checksums

print('=== Post-Review SHA-256 Checksum Audit ===')
stages_to_check = [
    ('raw_edfs', DATA_ROOT, baseline_snapshots.get('raw_edfs'), '*.edf'),
    ('phase1_audit', PHASE1_AUDIT_DIR, baseline_snapshots.get('phase1_audit'), '*'),
    ('phase2_signal_quality', PHASE2_OUTPUT_DIR, baseline_snapshots.get('phase2_signal_quality'), '*'),
    ('phase3a_diagnostic', PHASE3A_OUTPUT_DIR, baseline_snapshots.get('phase3a_diagnostic'), '*'),
    ('phase3b_preprocessing', PHASE3B_OUTPUT_DIR, baseline_snapshots.get('phase3b_preprocessing'), '*'),
]

for stage_name, stage_dir, baseline_dict, pat in stages_to_check:
    ver_res = verify_directory_checksums(
        baseline_hashes=baseline_dict,
        current_directory=stage_dir,
        stage_name=stage_name,
        pattern=pat,
    )
    if ver_res['status'] == 'verified_unmodified':
        print(f"[VERIFIED] {stage_name}: All {ver_res['n_files_checked']} files verified byte-for-byte identical via SHA-256.")
    elif ver_res['status'] == 'baseline_snapshot_unavailable':
        print(f"[LIMITATION] {stage_name}: {ver_res['detail']}")
    else:
        raise RuntimeError(f"Integrity violation in {stage_name}: {ver_res['detail']}")

report_path = REVIEW_OUTPUT_DIR / 'phase3b_review_report.md'
display(Markdown(report_path.read_text(encoding='utf-8')))

figures_dir = REVIEW_OUTPUT_DIR / 'figures'
png_files = sorted(p for p in figures_dir.glob('*.png') if p.is_file() and p.stat().st_size > 0)
for fig_file in png_files:
    print(f'Figure: {fig_file.name}')
    display(Image(filename=str(fig_file)))